# 🍄 슈퍼마리오 DQN 학습
**런타임 → 런타임 유형 변경 → T4 GPU 선택 후 실행하세요**

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/supermario_dl_project'
os.makedirs(f'{DRIVE_PATH}/models', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/results', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/videos', exist_ok=True)
print('Drive 마운트 완료!')

## 2. 패키지 설치

In [ ]:
!apt-get install -y xvfb cmake fonts-nanum > /dev/null 2>&1

!pip install --upgrade pip wheel -q
!pip install "setuptools>=65.5.1,<82" -q
!pip install nes-py==8.2.1 --no-build-isolation -q
!pip install gym==0.26.2 -q
!pip install gym-super-mario-bros==7.4.0 --no-deps -q
!pip install opencv-python-headless imageio imageio-ffmpeg pyvirtualdisplay tqdm -q

import glob, re, os, sys, numpy as np

# [한글 폰트] NanumGothic 절대경로로 직접 등록 (Colab 표준 경로)
import matplotlib
import matplotlib.font_manager as fm

_NANUM_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(_NANUM_PATH):
    fm.fontManager.addfont(_NANUM_PATH)
    matplotlib.rcParams['font.family'] = fm.FontProperties(fname=_NANUM_PATH).get_name()
    print(f'[폰트] NanumGothic 적용')
else:
    # 폰트 파일이 다른 경로에 있을 경우 탐색
    _found = glob.glob('/usr/share/fonts/**/*anum*.ttf', recursive=True)
    if _found:
        fm.fontManager.addfont(_found[0])
        matplotlib.rcParams['font.family'] = fm.FontProperties(fname=_found[0]).get_name()
        print(f'[폰트] {os.path.basename(_found[0])} 적용')
    else:
        print('[폰트] NanumGothic 미발견 — 한글이 깨질 수 있음')
matplotlib.rcParams['axes.unicode_minus'] = False

# [패치 0] numpy 런타임: np.bool8 직접 등록
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_
    print('[numpy] np.bool8 등록')

# [패치 1] nes_py, gym_super_mario_bros: numpy 2.0 uint8 오버플로우 종합 수정
files = (
    glob.glob('/usr/local/lib/python*/dist-packages/nes_py/*.py') +
    glob.glob('/usr/local/lib/python*/dist-packages/gym_super_mario_bros/*.py')
)
p1 = re.compile(r'(self\.ram\[[^\]]+\]|self\.\w+_size)\s*\*\s*((?:0x[0-9a-fA-F]+|\d+)(?:\*\*\d+)?)')
p2 = re.compile(r'(?<!int\()self\.ram\[([^\]:]+)\](?!\s*[=\[])')
total = 0
for path in files:
    with open(path) as f:
        content = f.read()
    patched = p1.sub(lambda m: f'int({m.group(1)}) * {m.group(2)}', content)
    patched = p2.sub(r'int(self.ram[\1])', patched)
    if patched != content:
        with open(path, 'w') as f:
            f.write(patched)
        for pyc in glob.glob(os.path.join(os.path.dirname(path), '__pycache__',
                             os.path.basename(path).replace('.py', '*.pyc'))):
            os.remove(pyc)
        print(f'  [uint8] {path.split("/")[-1]}')
        total += 1
print(f'uint8 패치: {total}개 파일')

# [패치 2] passive_env_checker: np.bool8 제거됨
for path in glob.glob('/usr/local/lib/python*/dist-packages/gym/utils/passive_env_checker.py'):
    with open(path) as f:
        content = f.read()
    patched = content.replace('np.bool8', 'np.bool_')
    if patched != content:
        with open(path, 'w') as f:
            f.write(patched)
        for pyc in glob.glob(os.path.join(os.path.dirname(path), '__pycache__', 'passive_env_checker*.pyc')):
            os.remove(pyc)
        print('[bool8] passive_env_checker.py 패치 완료')

# [패치 3] time_limit: old API(4-return) 호환
OLD = 'observation, reward, terminated, truncated, info = self.env.step(action)'
NEW = ('result = self.env.step(action)\n'
       '        if len(result) == 4:\n'
       '            observation, reward, terminated, info = result\n'
       '            truncated = False\n'
       '        else:\n'
       '            observation, reward, terminated, truncated, info = result')
for path in glob.glob('/usr/local/lib/python*/dist-packages/gym/wrappers/time_limit.py'):
    with open(path) as f:
        content = f.read()
    if OLD in content:
        with open(path, 'w') as f:
            f.write(content.replace(OLD, NEW))
        for pyc in glob.glob(os.path.join(os.path.dirname(path), '__pycache__', 'time_limit*.pyc')):
            os.remove(pyc)
        print('[time_limit] 4/5-return 호환 패치 완료')

# 모듈 캐시 초기화
cleared = [k for k in list(sys.modules.keys())
           if k.startswith(('gym', 'nes_py', 'gym_super_mario_bros'))]
for k in cleared:
    del sys.modules[k]
print(f'모듈 캐시 초기화: {len(cleared)}개')
print('설치 완료!')

## 3. 가상 디스플레이 시작 (Colab 헤드리스 렌더링)

In [ ]:
from pyvirtualdisplay import Display
display = Display(visible=0, size=(400, 300))
display.start()
print('가상 디스플레이 시작!')

## 4. GitHub에서 프로젝트 클론

In [ ]:
GITHUB_REPO = 'https://github.com/HyunDove/supermario-dqn.git'

import os
if not os.path.exists('/content/supermario-dqn'):
    !git clone {GITHUB_REPO} /content/supermario-dqn
else:
    !cd /content/supermario-dqn && git pull

os.chdir('/content/supermario-dqn')
print('프로젝트 준비 완료!')

## 5. GPU 확인

In [ ]:
import torch
print(f'GPU 사용 가능: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 모델: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')

## 6. 학습 실행 (영상 자동 기록 포함)

In [ ]:
import sys, os, json
sys.path.insert(0, '/content/supermario-dqn')

import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.env.wrappers import make_env
from src.agent.dqn_agent import DQNAgent
from src.utils.recorder import record_episode

# ====== 설정 ======
EPISODES = 1000
SAVE_EVERY = 200
RECORD_AT = [0, 1000, 5000, 7000]
CHECKPOINT_PATH = None
RECORD_STEPS = 450   # 영상 길이: 450스텝 = 30초 (15fps 기준)
RECORD_SPEED = 2.0   # 재생 배속: 1.0=실제속도, 2.0=2배속, 0.5=0.5배속
# ==================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
env = make_env(world=1, stage=1)
n_actions = env.action_space.n
agent = DQNAgent(n_actions=n_actions, device=device)

start_episode = 0
rewards_history = []
best_reward = -float('inf')

if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
    agent.load(CHECKPOINT_PATH)
    start_episode = int(CHECKPOINT_PATH.split('ep')[-1].split('.')[0])
    stats_path = f'{DRIVE_PATH}/results/rewards_history.json'
    if os.path.exists(stats_path):
        with open(stats_path) as f:
            rewards_history = json.load(f)
    print(f'에피소드 {start_episode}부터 재개')


def save_checkpoint_graph(rewards, episode, drive_path):
    """에피소드 체크포인트 시점의 학습 곡선 저장"""
    os.makedirs(f'{drive_path}/results', exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f'학습 곡선 — EP {episode} 시점', fontsize=13, fontweight='bold')

    ax1 = axes[0]
    ax1.plot(rewards, alpha=0.25, color='steelblue', linewidth=0.8, label='에피소드 보상')
    window = min(100, len(rewards))
    if len(rewards) >= window:
        avg = np.convolve(rewards, np.ones(window) / window, mode='valid')
        ax1.plot(range(window - 1, len(rewards)), avg,
                 color='tomato', linewidth=2, label=f'{window}회 이동 평균')
    ax1.axvline(x=episode - 1, color='gold', linestyle='--', alpha=0.9, linewidth=1.5)
    ax1.set_xlabel('에피소드')
    ax1.set_ylabel('총 보상')
    ax1.set_title('보상 변화')
    ax1.legend(fontsize=9)
    ax1.grid(alpha=0.3)

    ax2 = axes[1]
    ax2.axis('off')
    last100 = rewards[-100:] if len(rewards) >= 100 else rewards
    metrics = [
        ('현재 에피소드', f'{episode}'),
        ('총 학습 스텝', f'{agent.learn_step:,}'),
        ('현재 ε (epsilon)', f'{agent.epsilon:.4f}'),
        ('', ''),
        ('최근 100회 평균 보상', f'{np.mean(last100):.1f}'),
        ('최근 100회 최대 보상', f'{np.max(last100):.1f}'),
        ('최근 100회 최소 보상', f'{np.min(last100):.1f}'),
        ('', ''),
        ('전체 최고 보상', f'{max(rewards):.1f}'),
        ('전체 평균 보상', f'{np.mean(rewards):.1f}'),
    ]
    y = 0.95
    for label, value in metrics:
        if label == '':
            y -= 0.06
            continue
        ax2.text(0.05, y, label, transform=ax2.transAxes, fontsize=10, color='gray')
        ax2.text(0.65, y, value, transform=ax2.transAxes, fontsize=10, fontweight='bold', color='black')
        y -= 0.09
    ax2.set_title('지표 요약', fontsize=11)

    plt.tight_layout()
    path = f'{drive_path}/results/curve_ep{episode:04d}.png'
    plt.savefig(path, dpi=130, bbox_inches='tight')
    plt.close()
    print(f'  그래프 저장: {path}')


# --- EP0 영상 + 그래프 ---
if 0 in RECORD_AT and start_episode == 0:
    print('\n[EP 0] 학습 전 영상 기록 중...')
    reward_ep0, frames_ep0 = record_episode(
        lambda: make_env(1, 1), agent,
        f'{DRIVE_PATH}/videos/mario_ep0000.mp4',
        epsilon=1.0, target_steps=RECORD_STEPS, playback_speed=RECORD_SPEED
    )
    print(f'[EP 0] 영상 저장 완료 | 보상: {reward_ep0:.1f} | 프레임: {frames_ep0}')
    save_checkpoint_graph([reward_ep0], 0, DRIVE_PATH)

# --- 학습 루프 ---
for episode in tqdm(range(start_episode, EPISODES)):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.select_action(state)
        next_state, reward, done, info = env.step(action)
        agent.store(state, action, reward, next_state, done)
        agent.learn()
        state = next_state
        total_reward += reward

    rewards_history.append(total_reward)
    current_ep = episode + 1

    if current_ep % SAVE_EVERY == 0:
        agent.save(f'{DRIVE_PATH}/models/mario_ep{current_ep}.pth')
        with open(f'{DRIVE_PATH}/results/rewards_history.json', 'w') as f:
            json.dump(rewards_history, f)
        avg = np.mean(rewards_history[-100:])
        print(f'\n[EP {current_ep}] 평균보상: {avg:.1f} | epsilon: {agent.epsilon:.3f}')

    if total_reward > best_reward:
        best_reward = total_reward
        agent.save(f'{DRIVE_PATH}/models/mario_best.pth')

    if current_ep in RECORD_AT:
        saved_eps = agent.epsilon
        print(f'\n[EP {current_ep}] 영상·그래프 기록 중...')
        vid_reward, frames = record_episode(
            lambda: make_env(1, 1), agent,
            f'{DRIVE_PATH}/videos/mario_ep{current_ep:04d}.mp4',
            epsilon=0.05, target_steps=RECORD_STEPS, playback_speed=RECORD_SPEED
        )
        agent.epsilon = saved_eps
        print(f'[EP {current_ep}] 영상 저장 완료 | 보상: {vid_reward:.1f} | 프레임: {frames}')
        save_checkpoint_graph(rewards_history, current_ep, DRIVE_PATH)

env.close()
with open(f'{DRIVE_PATH}/results/rewards_history.json', 'w') as f:
    json.dump(rewards_history, f)
print('\n학습 완료!')

## 7. 학습 곡선 그래프 저장

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker
import numpy as np
import os

# 한글 폰트 (이 셀 단독 실행 시에도 적용)
_NANUM_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(_NANUM_PATH):
    fm.fontManager.addfont(_NANUM_PATH)
    plt.rcParams['font.family'] = fm.FontProperties(fname=_NANUM_PATH).get_name()
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('슈퍼마리오 DQN 학습 결과', fontsize=14, fontweight='bold')

ax1 = axes[0]
ax1.plot(rewards_history, alpha=0.3, color='steelblue', label='에피소드 보상')
window = min(100, len(rewards_history))
avg = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
ax1.plot(range(window-1, len(rewards_history)), avg, color='tomato', linewidth=2, label=f'{window}회 이동 평균')

record_points = [ep for ep in [0, 1000, 5000, 7000] if ep < len(rewards_history)]
for ep in record_points:
    ax1.axvline(x=ep, color='gold', linestyle='--', alpha=0.7, linewidth=1.2)
    ax1.text(ep, ax1.get_ylim()[1]*0.9, f'EP{ep}\n영상', fontsize=7, ha='center', color='goldenrod')

ax1.set_xlabel('에피소드')
ax1.set_ylabel('총 보상')
ax1.set_title('에피소드별 보상 변화')
ax1.legend()
ax1.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(range(window-1, len(rewards_history)), avg, color='tomato', linewidth=2)
ax2.fill_between(range(window-1, len(rewards_history)), avg, alpha=0.2, color='tomato')
ax2.set_xlabel('에피소드')
ax2.set_ylabel('평균 보상 (100회)')
ax2.set_title('학습 안정화 추세 (100회 이동 평균)')
ax2.grid(alpha=0.3)

plt.tight_layout()
save_path = f'{DRIVE_PATH}/results/training_curve.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'그래프 저장: {save_path}')

## 8. 저장된 파일 목록 확인

In [ ]:
import os

print('=== 저장된 모델 ===')
for f in sorted(os.listdir(f'{DRIVE_PATH}/models')):
    size = os.path.getsize(f'{DRIVE_PATH}/models/{f}') / 1024 / 1024
    print(f'  {f}  ({size:.1f} MB)')

print('\n=== 기록된 영상 ===')
for f in sorted(os.listdir(f'{DRIVE_PATH}/videos')):
    size = os.path.getsize(f'{DRIVE_PATH}/videos/{f}') / 1024 / 1024
    print(f'  {f}  ({size:.1f} MB)')

print('\n=== 결과 그래프 ===')
for f in sorted(os.listdir(f'{DRIVE_PATH}/results')):
    print(f'  {f}')

## 9. 세션 끊긴 후 이어서 학습하기

In [ ]:
# 1~5번 셀 재실행 후, 6번 셀에서 CHECKPOINT_PATH 수정
# 예시:
# CHECKPOINT_PATH = f'{DRIVE_PATH}/models/mario_ep1000.pth'

import os
print('마지막 저장 모델:')
models = sorted(os.listdir(f'{DRIVE_PATH}/models'))
for m in models[-3:]:
    print(f'  {DRIVE_PATH}/models/{m}')